# CreditGuard: Phase 2 - Model Training & Hyperparameter Tuning

This notebook covers the training, 5-fold stratified cross-validation, and hyperparameter optimization for three candidate models:
1. **Logistic Regression (Baseline)**: L2 regularized, class-weighted, scaled inputs.
2. **Random Forest**: Bagging ensemble, balanced subsample weighting.
3. **XGBoost**: Gradient boosting machine with `scale_pos_weight` class penalty.

### Validation Rules:
- Stratified 5-Fold Cross Validation evaluated with `roc_auc`.
- Complete training set is refit with the best hyperparameters.
- The test set is held out completely.

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
from src.config import RAW_DATA_PATH, TARGET_COL
from src.ingestion.loader import CreditDataLoader
from src.models.trainer import ModelTrainer

loader = CreditDataLoader(RAW_DATA_PATH)
raw_df = loader.load_raw_data().drop_duplicates()
clean_df = raw_df[(raw_df['person_age'] >= 18) & (raw_df['person_age'] <= 100)].copy()
emp_mask = ((clean_df['person_emp_length'] <= 60) & (clean_df['person_emp_length'] <= (clean_df['person_age'] - 14))) | clean_df['person_emp_length'].isna()
clean_df = clean_df[emp_mask].reset_index(drop=True)

trainer = ModelTrainer()
X_train, X_test, y_train, y_test = trainer.split_data(clean_df, target_col=TARGET_COL)
print(f'Training cohort: {X_train.shape[0]:,} loans | Baseline default rate: {y_train.mean():.2%}')

## 1. Candidate Model Setup & Search Grids

In [ ]:
candidates = trainer.get_candidate_models(y_train)
for name, cfg in candidates.items():
    model_cls = cfg['model'].__class__.__name__
    print(f'Model: {name} ({model_cls}) | Scale Numerics: {cfg["scale_numeric"]}')
    print(f'Search Grid: {list(cfg["params"].keys())}')

## 2. Cross-Validation Results

In [ ]:
training_results = trainer.train_all_models(X_train, y_train)
for name, res in training_results.items():
    print(f'[{name}]')
    print(f'  5-Fold CV ROC-AUC: {res.cv_mean_roc_auc:.4f} (+/- {res.cv_std_roc_auc:.4f})')
    print(f'  Optimal Params: {res.best_params}')